In [1]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
import os
from langgraph.graph import StateGraph, MessagesState, START


load_dotenv()  


model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.environ["DEEPSEEK_API_BASE"],
)

def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": response}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
graph = builder.compile()

In [2]:
input_message = {"role": "user", "content": "hi! 我是tomie"}
for chunk in graph.stream({"messages": [input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

input_message = {"role": "user", "content": "我叫什么名字?"}
for chunk in graph.stream({"messages": [input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi! 我是tomie
================================== Ai Message ==================================

Hi Tomie！很高兴认识你～这个名字让我想起伊藤润二笔下的那个神秘又迷人的角色呢😄 不过你先别急着点头，我猜你更想聊聊你自己？或是有什么特别的故事想分享？随便说，我在这儿听着呢～
================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

很抱歉，我无法知道你的名字。我们刚刚开始对话，你还没有告诉我呢！如果想让我用名字称呼你，可以随时告诉我哦～😊


In [4]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}
input_message = {"role": "user", "content": "hi! 我是tomie"}

for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
  chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi! 我是tomie
================================== Ai Message ==================================

你好，Tomie！👋  
提到你，我就想到了伊藤润二老师笔下那个经典的、让人又着迷又战栗的角色——永远不死的妖艳少女，带点诡异的美感。你是想聊聊这个角色，还是想玩点恐怖/悬疑风格的脑洞？或者…你只是单纯想打个招呼？😄  

不管怎样，我在这儿陪着你，不会用分尸的方式“复制”你哦～


In [6]:
input_message = {"role": "user", "content": "我叫什么名字?"}
for chunk in graph.stream(
    {"messages": [input_message]},
    {"configurable": {"thread_id": "1"}}, # different thread_id
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

你叫 **Tomie** 呀～刚才你自我介绍过的 😊  
是要考考我有没有认真听你说的话吗？


In [28]:
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()


in_memory_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(
            model="BAAI/bge-m3",
            api_key=os.environ.get("SILICONFLOW_API_KEY"),
            base_url="https://api.siliconflow.cn/v1",
            ),
        "dims": 1024,
    }
)

embeddings = OpenAIEmbeddings(
        model="BAAI/bge-m3",
            api_key=os.environ.get("SILICONFLOW_API_KEY"),
            base_url="https://api.siliconflow.cn/v1",
)

print(len(embeddings.embed_query("test")))

1024


In [31]:
import uuid
from typing import Annotated
from typing_extensions import TypedDict

from langchain_deepseek import ChatDeepSeek
import os
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore


model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)


# 注意：我们将 Store 参数传递给节点 --
# 这是我们编译图时使用的 Store
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    # 从存储中检索用户信息
    user_id = config["configurable"]["user_id"]
    # 从存储中检索用户信息
    namespace = ("memories", user_id)
    memories = store.search(namespace, query=str(state["messages"][-1].content))
    info = "\n".join([d.value["data"] for d in memories])
    system_msg = f"你是一个正在与用户交谈的小助手。用户信息：{info}"

    # 如果用户要求模型记住信息，则存储新的记忆
    last_message = state["messages"][-1]
    if "记住" in last_message.content.lower() or "remember" in last_message.content.lower():
        # 硬编码一个记忆
        memory = "用户名字是tomiezhang"
        store.put(namespace, str(uuid.uuid4()), {"data": memory})

    response = model.invoke(
        [{"role": "system", "content": system_msg}] + state["messages"]
    )
    return {"messages": response}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

# 注意：我们在编译图时传递了 store 对象
graph = builder.compile(checkpointer=MemorySaver(), store=in_memory_store)

In [32]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}
input_message = {"role": "user", "content": "请记住我的名字叫tomiezhang!"}
for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

请记住我的名字叫tomiezhang!
================================== Ai Message ==================================

好的，tomiezhang！我已经记住你的名字了，以后会这样称呼你。😊 有什么我可以帮你的吗？


In [26]:
! pip install -U pymongo langgraph langgraph-checkpoint-mongodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langgraph-checkpoint-mongodb]rch-utils]


In [27]:
import pymongo

# 创建MongoDB客户端连接
client = pymongo.MongoClient("mongodb://localhost:27017/")

# 测试连接
try:
    client.admin.command('ping')
    print("MongoDB连接成功！")
except Exception as e:
    print(f"MongoDB连接失败: {e}")


MongoDB连接成功！


In [34]:
from typing import Literal

from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent


@tool
def get_weather(city: Literal["北京", "深圳"]):
    """用来返回天气信息的工具函数。"""
    if city == "北京":
        return "北京天气晴朗 大约22度 湿度30%"
    elif city == "深圳":
        return "深圳天气多云 大约28度 湿度80%"
    else:
        raise AssertionError("Unknown city")


tools = [get_weather]
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)

from langgraph.checkpoint.mongodb import MongoDBSaver

MONGODB_URI = "localhost:27017"  # replace this with your connection string

with MongoDBSaver.from_conn_string(MONGODB_URI) as checkpointer:
    graph = create_agent(model, tools=tools, checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}
    response = graph.invoke(
        {"messages": [("human", "北京今天的天气如何？")]}, config
    )

In [35]:
print(response)

{'messages': [HumanMessage(content='北京今天的天气如何？', additional_kwargs={}, response_metadata={}, id='5024b84f-f9be-40c3-82d0-3fde8cd51598'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': '用户想知道北京今天的天气。我可以使用get_weather工具，城市参数为"北京"。'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 288, 'total_tokens': 352, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 288}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '26d83615-1a56-4afa-90f4-cf278eefc666', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7c89-9827-70f3-847a-7b0599857ef5-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'ca

In [36]:
from typing import Literal

from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

memory = MemorySaver()


@tool
def search(query: str):
    """调用此函数可以浏览网络。"""
    # 模拟一个网络搜索返回
    return "北京天气晴朗 大约22度 湿度30%"


tools = [search]
tool_node = ToolNode(tools)
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)

bound_model = model.bind_tools(tools)


def should_continue(state: MessagesState):
    """返回下一个要执行的节点。"""
    last_message = state["messages"][-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return END
    # 否则，如果有函数调用，我们继续
    return "action"


def filter_messages(messages: list):
    # 这是一个非常简单的辅助函数，它只使用最后一条消息
    return messages[-1:]


# 定义调用模型的函数
def call_model(state: MessagesState):
    messages = filter_messages(state["messages"])
    response = bound_model.invoke(messages)
    # 我们返回一个列表，因为这将被添加到现有列表中
    return {"messages": response}


# 定义一个新图
workflow = StateGraph(MessagesState)

# 定义我们将在其间循环的两个节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 将入口点设置为 `agent`
# 这意味着这个节点是第一个被调用的
workflow.add_edge(START, "agent")

# 现在添加一个条件边
workflow.add_conditional_edges(
    # 首先，我们定义起始节点。我们使用 `agent`。
    # 这意味着这些是在调用 `agent` 节点后采取的边。
    "agent",
    # 接下来，我们传入将确定下一个调用哪个节点的函数。
    should_continue,
    # 接下来，我们传入路径图 - 此边可能去往的所有可能节点
    ["action", END],
)

# 现在我们从 `action` 到 `agent` 添加一个普通边。
# 这意味着在调用 `action` 之后，下一步调用 `agent` 节点。
workflow.add_edge("action", "agent")

# 最后，我们编译它！
# 这将它编译成一个 LangChain Runnable，
# 意味着你可以像使用任何其他 runnable 一样使用它
app = workflow.compile(checkpointer=memory)

In [37]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "2"}}
input_message = HumanMessage(content="hi! 我是tomie")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

# 请注意，我们在这里使用了一个辅助函数，它只使用最后一条消息
# 这将导致我们的模型只看到最后一条消息
input_message = HumanMessage(content="我叫什么名字?")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

hi! 我是tomie
================================== Ai Message ==================================

Tomie 你好呀！👋 很高兴见到你！有什么我可以帮你的吗？或者想聊点什么？😊
================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

我并不知道你的名字哦！我们才刚开始对话，我还没有关于你的个人信息呢。😊

你可以告诉我你想让我怎么称呼你，或者直接告诉我你的名字，这样我就能在后续的对话中更好地和你交流啦！


In [38]:
from typing import Literal

from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage, RemoveMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END

memory = MemorySaver()


# 我们将添加一个`summary`属性（除了MessagesState已有的`messages`键之外）
class State(MessagesState):
    summary: str


# 我们将使用这个模型进行对话和总结
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)


# 定义调用模型的逻辑
def call_model(state: State):
    # 如果存在摘要，我们将其作为系统消息添加
    summary = state.get("summary", "")
    if summary:
        system_message = f"之前对话的摘要: {summary}"
        messages = [SystemMessage(content=system_message)] + state["messages"]
    else:
        messages = state["messages"]
    response = model.invoke(messages)
    # 我们返回一个列表，因为这将被添加到现有列表中
    return {"messages": [response]}


# 现在我们定义确定是结束还是总结对话的逻辑
def should_continue(state: State) -> Literal["summarize_conversation", END]:
    """返回下一个要执行的节点。"""
    messages = state["messages"]
    # 如果消息超过六条，则我们总结对话
    if len(messages) > 6:
        return "summarize_conversation"
    # 否则我们可以直接结束
    return END


def summarize_conversation(state: State):
    # 首先，我们总结对话
    summary = state.get("summary", "")
    if summary:
        # 如果已经存在摘要，我们使用不同的系统提示来总结它
        # 与没有摘要的情况不同
        summary_message = (
            f"这是迄今为止对话的摘要: {summary}\n\n"
            "考虑上面的新消息，扩展摘要:"
        )
    else:
        summary_message = "创建上述对话的摘要:"

    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    # 现在我们需要删除我们不再想显示的消息
    # 我将删除除最后两条以外的所有消息，但你可以更改这一点
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}


# 定义一个新图
workflow = StateGraph(State)

# 定义对话节点和总结节点
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 将入口点设置为对话
workflow.add_edge(START, "conversation")

# 现在添加一个条件边
workflow.add_conditional_edges(
    # 首先，我们定义起始节点。我们使用`conversation`。
    # 这意味着这些是在调用`conversation`节点后采取的边。
    "conversation",
    # 接下来，我们传入将确定下一个调用哪个节点的函数。
    should_continue,
)

# 现在我们从`summarize_conversation`到END添加一个普通边。
# 这意味着在调用`summarize_conversation`之后，我们结束。
workflow.add_edge("summarize_conversation", END)

# 最后，我们编译它！
app = workflow.compile(checkpointer=memory)

In [41]:
def print_update(update):
    for k, v in update.items():
        for m in v["messages"]:
            m.pretty_print()
        if "summary" in v:
            print(v["summary"])

In [40]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "4"}}
input_message = HumanMessage(content="hi! 我是tomie")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

input_message = HumanMessage(content="我叫什么名字?")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

input_message = HumanMessage(content="我喜欢AI应用开发!")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

================================ Human Message =================================

hi! 我是tomie
================================== Ai Message ==================================

Hi Tomie! 👋 It’s nice to meet you. How can I help you today? If you’re referencing the character from Junji Ito’s manga—haha, you certainly have a memorable name!
================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

你刚刚告诉我你叫 **Tomie** 呀～😊 不过如果你愿意再告诉我一次，或者想换个名字，我也很乐意记住新名字！
================================ Human Message =================================

我喜欢AI应用开发!
================================== Ai Message ==================================

太棒了！AI应用开发是一个非常有趣且充满潜力的领域 🚀  

如果你喜欢动手实践，可以试试这些方向：  
- **RAG（检索增强生成）**：将文档知识库与LLM结合，构建能回答私有问题的智能问答系统。  
- **Agent与工具调用**：让AI自动执行任务（比如查询天气、发送邮件）。  
- **多模态应用**：结合图片、音频处理能力，比如图像理解或语音助手。  

你最近有在玩什么具体的项目吗？比如用LangChain、AutoGPT，或者自己微调模型？如果有想讨论的技术细节或遇到的bug，我很乐意一起

In [42]:
values = app.get_state(config).values
values

{'messages': [HumanMessage(content='hi! 我是tomie', additional_kwargs={}, response_metadata={}, id='205fde2a-f0ab-4c28-acc6-f7e6f7c47829'),
  AIMessage(content='Hi Tomie! 👋 It’s nice to meet you. How can I help you today? If you’re referencing the character from Junji Ito’s manga—haha, you certainly have a memorable name!', additional_kwargs={'refusal': None, 'reasoning_content': '哦，用户说自己是tomie。这个名字有点特别，可能是日本恐怖漫画角色，也可能只是普通的英文名。需要先确认用户意图。如果是角色名，可能想玩角色扮演或者讨论相关作品。如果是真名，那就正常打招呼。\n\n考虑到对话刚开始，信息太少，最稳妥的方式是先礼貌回应，用表情符号营造友好氛围，同时留下开放选项让对方继续说明。可以提一下漫画角色Tomie作为试探，如果用户有兴趣会主动接话。\n\n用波浪线和emoji保持轻松感，避免直接追问显得太唐突。类似“I remember a character”这样的模糊提示，既点了可能性又不给压力。'}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 10, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 134, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0

In [43]:
input_message = HumanMessage(content="我更喜欢Python!")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

================================ Human Message =================================

我更喜欢Python!
================================== Ai Message ==================================

Python 绝对是 AI 应用开发的“利器” 🔥 从数据预处理、模型训练到部署上线，Python 的生态几乎覆盖了所有环节。  

你平时主要用哪些库呢？  
- **LangChain / LlamaIndex**（构建 RAG 或 Agent）  
- **Hugging Face Transformers**（微调或调用开源模型）  
- **FastAPI / Flask**（部署 API）  
- **Streamlit / Gradio**（快速搭建 Demo）  

或者你更偏向底层一点，比如用 PyTorch / JAX 自己搭模型？如果有正在做的小项目或者想尝试的方向，可以分享一下～ 说不定能一起碰撞出些新点子 😄
================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


以下是我

In [44]:
values = app.get_state(config).values
values

{'messages': [HumanMessage(content='我更喜欢Python!', additional_kwargs={}, response_metadata={}, id='89aa3d97-3b12-4fc7-b25a-cbaaff375a37'),
  AIMessage(content='Python 绝对是 AI 应用开发的“利器” 🔥 从数据预处理、模型训练到部署上线，Python 的生态几乎覆盖了所有环节。  \n\n你平时主要用哪些库呢？  \n- **LangChain / LlamaIndex**（构建 RAG 或 Agent）  \n- **Hugging Face Transformers**（微调或调用开源模型）  \n- **FastAPI / Flask**（部署 API）  \n- **Streamlit / Gradio**（快速搭建 Demo）  \n\n或者你更偏向底层一点，比如用 PyTorch / JAX 自己搭模型？如果有正在做的小项目或者想尝试的方向，可以分享一下～ 说不定能一起碰撞出些新点子 😄', additional_kwargs={'refusal': None, 'reasoning_content': '我们注意到用户说“我更喜欢Python”，这是在回应之前的对话。用户之前说喜欢AI应用开发，现在更具体到Python。作为AI助手，应该表示认可并展开讨论Python在AI开发中的优势，询问用户具体的方向或项目。'}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 258, 'total_tokens': 460, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 52, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_t

In [45]:
input_message = HumanMessage(content="我叫什么名字?")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

你的名字是 **Tomie** 😄
